# 08 — Conclusion & Reproducibility


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:

import json
import platform
import tensorflow as tf
import pyspark
import nibabel
import nilearn
import sklearn
from neuro.bids import validate_bids
from neuro.mlflow_utils import start_run

with start_run("08_conclusion"):
    report = validate_bids()
    summary = {
        "dataset": "ds000171",
        "subjects": report["n_subjects"],
        "mdd": report["n_mdd"],
        "nd": report["n_nd"],
        "runs_available": report["n_runs_available"],
        "tr_sec": report["tr_json_music"],
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "pyspark": pyspark.__version__,
        "sklearn": sklearn.__version__,
        "nibabel": nibabel.__version__,
        "nilearn": nilearn.__version__,
    }
    for k, v in summary.items():
        if isinstance(v, (int, float, str)):
            mlflow.log_param(k, v)
    print(json.dumps(summary, indent=2))


In [ ]:

print("""
## Summary
- BIDS ds000171: emotional music/non-music fMRI in MDD vs controls
- Pipeline: Spark ingestion -> Nilearn preprocessing -> ROI features -> TF transformer
- Visualisation: inline seaborn/matplotlib in notebooks (no HTML/PNG export)
- Tracking: mlflow experiments in mlruns/
- Limitations: small N=39; cross-subject CV essential; TR=3s
""")
